# S2C (From SAM to CAMs) — ResNet-101 + LayerCAM trên Google Colab

Notebook này triển khai và kiểm thử phương pháp **S2C (From SAM to CAMs, CVPR 2024)** trên backbone **`resnet101`** (`layer4 stride=1`, ảnh `448×448`) kết hợp **LayerCAM** (`layer2 + layer3 + layer4`) trên tập **Pascal VOC 2012**.

Khác với pipeline `CAM + SAM` trước đây (chỉ dùng SAM ở bước hậu xử lý khi inference), **S2C đưa tri thức biên của SAM quay ngược vào quá trình huấn luyện classifier**:

1. **SSC (SAM-Segment Contrasting — Feature-level)**:
   - Dùng `SamAutomaticMaskGenerator` (AMG) tạo bản đồ phân vùng class-agnostic cho ảnh train (cache 1 lần lên ổ đĩa / Google Drive).
   - Tính prototype cho từng SAM segment trên feature map `layer4` (`28×28`) và tối ưu **InfoNCE contrastive loss** để các pixel cùng segment gom cụm chặt và tách biệt biên vật thể.
2. **CPM (CAM-based Prompting Module — Logit-level, dùng LayerCAM)**:
   - Sau giai đoạn warmup (`CPM_START_EPOCH`), dùng **LayerCAM** (`layer2 + layer3 + layer4`) sinh bản đồ nhiệt và prompt `SamPredictor` tạo pseudo-mask chất lượng cao trên tập train.
   - Giám sát trực tiếp dense classifier logits của `ResNet-101` bằng pixel Cross-Entropy (có lọc vùng mơ hồ `IGNORE_INDEX = 255`).
3. **So sánh đối chứng (Baseline vs S2C)**:
   - So sánh trực tiếp **`resnet101_baseline`** (BCE thuần) và **`resnet101_s2c`** (`BCE + SSC + LayerCAM-CPM`) trên cùng **1,449 ảnh `Segmentation/val.txt`** ở độ phân giải gốc qua 4 phương pháp: `Vanilla CAM`, `CAM + SAM`, `LayerCAM`, `LayerCAM + SAM`.


## 1. Cấu hình thí nghiệm

- `REPO_BRANCH = "s2c-resnet101-layercam"`: branch chứa module `src/s2c.py` và notebook này.
- `TRAIN_SPLIT = "main_trainval"` (10,418 ảnh, không trùng `Seg/val`, chừa 313 ảnh `Seg/train` cho Optuna). Nếu muốn chạy thử nhanh trên Colab T4 (~10-15 phút), có thể đặt `TRAIN_SPLIT = "seg_train"` (1,464 ảnh) hoặc đặt `S2C_MAX_TRAIN_IMAGES = 2000`.
- `SAM_MODEL_TYPE = "vit_b"`: cân bằng tốt tốc độ cache AMG/CPM và VRAM trên Colab T4/L4 (có thể đổi `"vit_l"` nếu dùng L4/A100).


In [ ]:
# =====================================================================
# 1. CẤU HÌNH TỔNG THỂ — S2C + RESNET-101 + LAYERCAM
# =====================================================================
MODEL_NAME = "resnet101"
IMAGE_SIZE = 448
LAYER4_STRIDE_1 = True            # layer4[0] stride=(1,1) -> feature map 28x28 @448
FEAT_SIZE = 28 if LAYER4_STRIDE_1 else 14

TRAIN_SPLIT = "main_trainval"     # "main_trainval" (10,418) hoặc "seg_train" (1,464 - chạy nhanh)
S2C_MAX_TRAIN_IMAGES = None       # Đặt int (vd 2000) nếu muốn test nhanh, None = dùng toàn bộ split

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 15
LR_SCHEDULER = "poly"
POLY_POWER = 0.9
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP = True
SEED = 42
NUM_WORKERS = 2
REQUIRE_GPU = True

# --- S2C: SSC (SAM-Segment Contrasting) ---
LAMBDA_SSC = 0.20
SSC_TEMPERATURE = 0.10
S2C_PROJ_DIM = 256
AMG_POINTS_PER_SIDE = 16          # 16 (~4x nhanh hơn 32, rất hợp Colab); đổi 32 nếu muốn dày hơn
AMG_POINTS_PER_BATCH = 128

# --- S2C: CPM (CAM-based Prompting Module với LayerCAM) ---
LAMBDA_CPM = 0.50
CPM_START_EPOCH = 5               # Warmup 4 epoch đầu với BCE + SSC rồi mới sinh CPM pseudo-mask
CPM_REFRESH_EPOCHS = (5, 10)      # Các epoch đầu vào sẽ tạo/cập nhật CPM pseudo-mask
CPM_CAM_METHOD = "layercam"       # "layercam" hoặc "cam"
CPM_CAM_SCALES = (1.0,)           # Single-scale khi sinh CPM trên tập train để chạy nhanh
CPM_CAM_HFLIP = False
CPM_AMBIGUOUS_CAM_THR = 0.50      # Pixel SAM=0 nhưng CAM >= 0.50 -> gán IGNORE_INDEX (255)
CPM_SAM_PARAMS = {
    "cam_threshold": 0.22,
    "low_threshold": 0.10,
    "bg_threshold": 0.04,
    "use_neg_points": True,
    "box_pad": 20,
}

# --- CAM / LayerCAM lúc Tune & Eval ---
CAM_SCALES = (0.75, 1.0, 1.25)
CAM_HFLIP = True
LAYERCAM_LAYERS = ("layer2", "layer3", "layer4")
LAYERCAM_SHALLOW_GAMMA = 2.0
CLS_THRESHOLD = 0.5
USE_NEG_POINTS = True

# --- Optuna ---
CAM_TUNE_MAX_IMAGES = 260
TUNE_SUBSET_SIZE = 90
OPTUNA_CLS_TRIALS = 51
OPTUNA_CAM_TRIALS = 35
OPTUNA_SAM_TRIALS = 60
OPTUNA_LAYERCAM_TRIALS = 35
OPTUNA_LAYERCAM_SAM_TRIALS = 60
TUNE_PROTOCOL_VERSION = 1

# --- SAM ---
SAM_MODEL_TYPE = "vit_b"          # "vit_b" (~375MB) nhanh cho AMG+CPM; "vit_l" (~1.25GB) nặng hơn

# --- Quản lý Run / Checkpoint / GitHub ---
EXPERIMENT_NAME = "S2C_ResNet101_LayerCAM"
BASELINE_RUN_NAME = "PaperWSSS_3Backbones_MainTrainVal_448_Stride1"
TRAIN_BASELINE_IF_MISSING = True  # Nếu chưa có baseline resnet101/best.pt thì train để đối chứng
RESUME_IF_EXISTS = True
RETRAIN_S2C = False

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS331_WSSS"
REPO_URL = "https://github.com/minhthanh224/CS331-Project.git"
REPO_BRANCH = "s2c-resnet101-layercam"

METHODS = ["Vanilla CAM", "CAM + SAM", "LayerCAM", "LayerCAM + SAM"]

assert MODEL_NAME == "resnet101"
assert TRAIN_SPLIT in {"main_trainval", "seg_train"}
assert SAM_MODEL_TYPE in {"vit_b", "vit_l", "vit_h"}
assert CPM_CAM_METHOD in {"cam", "layercam"}
assert 1 <= CPM_START_EPOCH <= NUM_EPOCHS + 1
assert 0 < TUNE_SUBSET_SIZE <= CAM_TUNE_MAX_IMAGES


## 2. Thiết lập môi trường Colab / Local, cài `segment-anything` và đồng bộ GitHub branch


In [ ]:
import gc
import json
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "optuna", "scikit-learn",
            "git+https://github.com/facebookresearch/segment-anything.git",
        ],
        check=True,
    )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DRIVE_BASE = Path(DRIVE_PROJECT_DIR)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    PROJECT_ROOT = Path("/content/CS331-Project")
    if not (PROJECT_ROOT / ".git").exists():
        if PROJECT_ROOT.exists():
            shutil.rmtree(PROJECT_ROOT)
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_BRANCH], check=False)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "origin", REPO_BRANCH], check=False)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name in ("notebooks", "test"):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DRIVE_BASE = PROJECT_ROOT

project_root = PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torchvision.datasets.utils import check_integrity, download_url, extract_archive
from tqdm.auto import tqdm

from src import (
    SAM_MODEL_SPECS,
    VOC_CLASSES as CLASSES,
    S2CProjectionHead,
    SegmentationEvaluator,
    VOCClassificationDataset,
    VOCS2CDataset,
    build_balanced_tune_table,
    build_classifier,
    build_optimizer,
    build_optuna_pipeline,
    build_s2c_optimizer,
    build_sam_amg,
    build_sam_predictor,
    build_transforms,
    build_voc_label_table,
    cache_cams,
    cache_sam_segments,
    cls_summary_row,
    compute_classification_metrics,
    ensure_sam_checkpoint,
    evaluate_segmentation,
    gen_cam_pseudo_mask,
    gen_sam_pseudo_mask,
    generate_cpm_pseudo_masks,
    get_device,
    get_sam_embedding,
    image_cell,
    load_classifier,
    load_eval_sample,
    load_eval_samples,
    log_eval_results,
    make_cam_fn,
    make_loader,
    mask_cell,
    plot_grid,
    poly_scheduler,
    predict_probs,
    predict_sample_masks,
    read_split_ids,
    read_wsss_train_ids,
    seg_summary_row,
    set_sam_embedding,
    set_seed,
    train_one_epoch,
    train_s2c_one_epoch,
)

set_seed(SEED)
device = get_device(require_gpu=REQUIRE_GPU)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DRIVE_BASE   : {DRIVE_BASE}")
print(f"DEVICE       : {device} ({gpu_name})")


## 3. Chuẩn bị VOC2012, SAM checkpoint và thư mục lưu kết quả

- Trên Colab, SAM AMG segment maps được ghi vào SSD local (`/content/s2c_cache/...`) để tránh độ trễ ghi hàng nghìn file nhỏ lên Google Drive FUSE, sau đó nén thành 1 file `.tar` đồng bộ lên Drive (`archives/`) để các phiên chạy sau khôi phục trong vài giây.


In [ ]:
drive_run_dir = DRIVE_BASE / "runs" / EXPERIMENT_NAME
run_dir = drive_run_dir if IN_COLAB else (project_root / "runs" / EXPERIMENT_NAME)
ckpt_dir = run_dir / "checkpoints"
plots_dir = run_dir / "plots"
optuna_dir = run_dir / "optuna"
for d in (run_dir, ckpt_dir, plots_dir, optuna_dir):
    d.mkdir(parents=True, exist_ok=True)
for tag in ("resnet101_s2c", "resnet101_baseline"):
    (ckpt_dir / tag).mkdir(parents=True, exist_ok=True)

cache_root = Path("/content/s2c_cache") if IN_COLAB else (project_root / "runs" / EXPERIMENT_NAME / "cache")
sam_seg_dir = cache_root / f"sam_amg_{SAM_MODEL_TYPE}_pps{AMG_POINTS_PER_SIDE}_{TRAIN_SPLIT}"
cpm_mask_dir = cache_root / f"cpm_{CPM_CAM_METHOD}_{SAM_MODEL_TYPE}"
sam_seg_dir.mkdir(parents=True, exist_ok=True)
cpm_mask_dir.mkdir(parents=True, exist_ok=True)

archive_name = "VOCtrainval_11-May-2012.tar"
archive_md5 = "6cd6e144f989b92b3379bac3b3de84fd"

if IN_COLAB:
    archive_dir = DRIVE_BASE / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)

    data_root = Path("/content/data")
    data_root.mkdir(parents=True, exist_ok=True)
    voc_root = data_root / "VOCdevkit" / "VOC2012"
    local_archive = data_root / archive_name
    drive_archive = archive_dir / archive_name

    if not voc_root.is_dir():
        if not check_integrity(local_archive, archive_md5):
            if drive_archive.is_file():
                print("Copy VOC2012 archive từ Google Drive...")
                shutil.copy2(drive_archive, local_archive)
            else:
                urls = [
                    "https://data.pjreddie.com/files/" + archive_name,
                    "https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/" + archive_name,
                ]
                for url in urls:
                    try:
                        print(f"Đang tải VOC2012 từ {url} ...")
                        download_url(url, str(data_root), filename=archive_name, md5=archive_md5)
                        break
                    except Exception as exc:
                        print(f"Tải thất bại: {url} ({exc})")
                else:
                    raise RuntimeError("Không tải được VOC2012. Hãy upload archive lên Google Drive.")

        if not check_integrity(local_archive, archive_md5):
            raise RuntimeError("VOC2012 archive sai MD5.")
        extract_archive(str(local_archive), str(data_root))
        if not drive_archive.exists():
            shutil.copy2(local_archive, drive_archive)

    local_sam_ckpt = ensure_sam_checkpoint(SAM_MODEL_TYPE, data_root, cache_dirs=[archive_dir])
    drive_sam_ckpt = archive_dir / SAM_MODEL_SPECS[SAM_MODEL_TYPE][0]
    if not drive_sam_ckpt.is_file():
        shutil.copy2(local_sam_ckpt, drive_sam_ckpt)
else:
    archive_dir = project_root / "checkpoints"
    voc_root = project_root / "data" / "VOC2012"
    local_sam_ckpt = ensure_sam_checkpoint(SAM_MODEL_TYPE, project_root / "checkpoints")

sam_seg_archive_name = f"sam_amg_{SAM_MODEL_TYPE}_pps{AMG_POINTS_PER_SIDE}_{TRAIN_SPLIT}.tar"
drive_sam_seg_tar = (archive_dir / sam_seg_archive_name) if IN_COLAB else (cache_root / sam_seg_archive_name)

print("VOC2012     :", voc_root)
print("SAM ckpt    :", local_sam_ckpt)
print("Run dir     :", run_dir)
print("SAM seg dir :", sam_seg_dir)
print("CPM dir     :", cpm_mask_dir)


## 4. Chia tập dữ liệu (`Main/trainval \ Seg/val`, `Seg/train`, `Seg/val`)


In [ ]:
seg_train_ids = read_split_ids(voc_root, "Segmentation", "train")
val_ids = read_split_ids(voc_root, "Segmentation", "val")
train_ids = read_wsss_train_ids(voc_root, TRAIN_SPLIT)
if S2C_MAX_TRAIN_IMAGES is not None:
    train_ids = train_ids[: int(S2C_MAX_TRAIN_IMAGES)]

assert len(seg_train_ids) == 1464 and len(val_ids) == 1449
assert set(train_ids).isdisjoint(val_ids) and set(seg_train_ids).isdisjoint(val_ids)

df_train = build_voc_label_table(voc_root, train_ids, CLASSES)
df_seg_train = build_voc_label_table(voc_root, seg_train_ids, CLASSES)
df_val = build_voc_label_table(voc_root, val_ids, CLASSES)
df_tune_balanced = build_balanced_tune_table(
    df_seg_train, train_ids, classes=CLASSES, max_images=CAM_TUNE_MAX_IMAGES, seed=SEED
)

print(f"1. Train ({TRAIN_SPLIT})     : {len(df_train):5d} ảnh")
print(f"2. Seg/train (pool tune)     : {len(df_seg_train):5d} ảnh")
print(f"   -> Tune cân bằng (Optuna) : {len(df_tune_balanced):5d} ảnh")
print(f"3. Eval (Seg/val)            : {len(df_val):5d} ảnh")

eval_transform = build_transforms(IMAGE_SIZE, train=False)
train_transform_base = build_transforms(IMAGE_SIZE, train=True)

base_train_dataset = VOCClassificationDataset(voc_root, df_train, CLASSES, train_transform_base)
s2c_train_dataset = VOCS2CDataset(
    voc_root,
    df_train,
    CLASSES,
    image_size=IMAGE_SIZE,
    feat_size=FEAT_SIZE,
    sam_seg_dir=sam_seg_dir,
    cpm_mask_dir=cpm_mask_dir,
    train=True,
)
seg_train_dataset = VOCClassificationDataset(voc_root, df_seg_train, CLASSES, eval_transform)
tune_balanced_dataset = VOCClassificationDataset(voc_root, df_tune_balanced, CLASSES, eval_transform)
val_dataset = VOCClassificationDataset(voc_root, df_val, CLASSES, eval_transform)

seg_train_loader = make_loader(seg_train_dataset, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device)
val_loader = make_loader(val_dataset, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device)
criterion = nn.BCEWithLogitsLoss()


## 5. Bước SSC Cache — Sinh / khôi phục SAM Automatic Segment Maps

Chạy `SamAutomaticMaskGenerator` đúng **1 lần** cho các ảnh trong `train_ids`. Nếu trên Google Drive đã có file `.tar` đã cache trước đó, notebook tự giải nén về SSD local mà không phải chạy lại.


In [ ]:
from segment_anything import sam_model_registry

# Khôi phục cache SAM AMG từ Drive nếu local chưa đủ file
existing_seg_pngs = len(list(sam_seg_dir.glob("*.png")))
if existing_seg_pngs < len(train_ids) and IN_COLAB and drive_sam_seg_tar.is_file():
    print(f"Giải nén SAM AMG cache từ Drive: {drive_sam_seg_tar} ...")
    with tarfile.open(drive_sam_seg_tar, "r") as tf:
        tf.extractall(sam_seg_dir)
    existing_seg_pngs = len(list(sam_seg_dir.glob("*.png")))
    print(f"Đã khôi phục {existing_seg_pngs}/{len(train_ids)} segment maps.")

sam_model = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(local_sam_ckpt)).to(device=device).eval()

if existing_seg_pngs < len(train_ids):
    amg = build_sam_amg(
        sam_model,
        points_per_side=AMG_POINTS_PER_SIDE,
        points_per_batch=AMG_POINTS_PER_BATCH,
    )
    amg_stats = cache_sam_segments(
        voc_root,
        train_ids,
        amg=amg,
        output_dir=sam_seg_dir,
        desc=f"Caching SAM AMG ({SAM_MODEL_TYPE}, pps={AMG_POINTS_PER_SIDE})",
    )
    print("AMG cache stats:", amg_stats)
    del amg
    gc.collect()
    torch.cuda.empty_cache()

    if IN_COLAB and amg_stats["generated"] > 0:
        print(f"Đang đóng gói SAM AMG cache lên Drive: {drive_sam_seg_tar} ...")
        with tarfile.open(drive_sam_seg_tar, "w") as tf:
            for png_path in sorted(sam_seg_dir.glob("*.png")):
                tf.add(png_path, arcname=png_path.name)
        print("Đã lưu SAM AMG cache lên Drive.")
else:
    print(f"[Cache OK] Đã đủ {existing_seg_pngs}/{len(train_ids)} SAM segment maps.")

# Tái sử dụng sam_model đang trên GPU cho SamPredictor (CPM + Optuna + Eval)
from segment_anything import SamPredictor
predictor = SamPredictor(sam_model)
print(f"SamPredictor ({SAM_MODEL_TYPE}) đã sẵn sàng trên {device}")


## 6. Huấn luyện `ResNet-101` với S2C (`BCE + SSC + LayerCAM-CPM`) & chuẩn bị Baseline

- **Epochs `1 .. CPM_START_EPOCH - 1`**: Warmup với `BCEWithLogitsLoss` + `lambda_ssc * L_SSC`.
- **Tại các epoch trong `CPM_REFRESH_EPOCHS` (mặc định epoch `5` và `10`)**:
  Dùng `model` hiện tại + **LayerCAM** (`layer2, layer3, layer4`) + `SamPredictor` sinh bộ CPM pseudo-mask mới lưu vào `cpm_mask_dir`.
- **Epochs `CPM_START_EPOCH .. NUM_EPOCHS`**: Huấn luyện đồng thời `L_CLS + lambda_ssc * L_SSC + lambda_cpm * L_CPM`.
- Checkpoint `best.pt` ưu tiên chọn theo `val_mAP` tốt nhất **từ khi CPM đã kích hoạt** (`epoch >= CPM_START_EPOCH`).


In [ ]:
config_record = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "layer4_stride1": LAYER4_STRIDE_1,
    "train_split": TRAIN_SPLIT,
    "train_num_images": len(train_ids),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler": LR_SCHEDULER,
    "poly_power": POLY_POWER,
    "lambda_ssc": LAMBDA_SSC,
    "ssc_temperature": SSC_TEMPERATURE,
    "s2c_proj_dim": S2C_PROJ_DIM,
    "amg_points_per_side": AMG_POINTS_PER_SIDE,
    "lambda_cpm": LAMBDA_CPM,
    "cpm_start_epoch": CPM_START_EPOCH,
    "cpm_refresh_epochs": list(CPM_REFRESH_EPOCHS),
    "cpm_cam_method": CPM_CAM_METHOD,
    "cpm_cam_scales": list(CPM_CAM_SCALES),
    "cpm_cam_hflip": CPM_CAM_HFLIP,
    "cpm_ambiguous_cam_thr": CPM_AMBIGUOUS_CAM_THR,
    "cam_scales": list(CAM_SCALES),
    "cam_hflip": CAM_HFLIP,
    "layercam_layers": list(LAYERCAM_LAYERS),
    "layercam_shallow_gamma": LAYERCAM_SHALLOW_GAMMA,
    "use_neg_points": USE_NEG_POINTS,
    "sam_model": SAM_MODEL_TYPE,
    "tune_protocol_version": TUNE_PROTOCOL_VERSION,
    "eval_protocol": "original_resolution",
    "seed": SEED,
}
(run_dir / "config.json").write_text(json.dumps(config_record, indent=2, ensure_ascii=False), encoding="utf-8")


def _find_baseline_ckpt():
    candidates = [
        ckpt_dir / "resnet101_baseline" / "best.pt",
        DRIVE_BASE / "runs" / BASELINE_RUN_NAME / "checkpoints" / "resnet101" / "best.pt",
        project_root / "checkpoints" / "resnet101" / "best.pt",
    ]
    for cand in candidates:
        if cand.is_file():
            return cand
    return None


def _load_history(ckpt_path: Path):
    try:
        payload = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        if isinstance(payload, dict):
            return payload.get("history", [])
    except Exception:
        pass
    return []


trained_models = {}
all_histories = {}

# ---------------------------------------------------------------------
# 6A. S2C ResNet-101
# ---------------------------------------------------------------------
s2c_dir = ckpt_dir / "resnet101_s2c"
s2c_best_path = s2c_dir / "best.pt"
s2c_weights_path = s2c_dir / "best_weights_only.pt"

if RESUME_IF_EXISTS and not RETRAIN_S2C and s2c_best_path.is_file():
    trained_models["resnet101_s2c"] = load_classifier(
        MODEL_NAME, s2c_best_path, device=device, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    )
    all_histories["resnet101_s2c"] = _load_history(s2c_best_path)
    print(f"[Resume] Nạp S2C ResNet-101 từ {s2c_best_path} ({len(all_histories['resnet101_s2c'])} epochs)")
else:
    s2c_loader = make_loader(
        s2c_train_dataset, BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, device=device, seed=SEED,
    )
    s2c_model = build_classifier(
        model_name=MODEL_NAME, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    ).to(device)
    proj_head = S2CProjectionHead(in_channels=2048, proj_dim=S2C_PROJ_DIM).to(device)
    optimizer = build_s2c_optimizer(
        s2c_model, proj_head, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = poly_scheduler(
        optimizer, total_steps=NUM_EPOCHS * len(s2c_loader), power=POLY_POWER
    )

    history = []
    best_s2c_map = -1.0
    best_any_map = -1.0
    best_epoch = 0

    print("=" * 88)
    print(
        f"TRAIN S2C RESNET-101 | {NUM_EPOCHS} epochs | split={TRAIN_SPLIT} ({len(s2c_train_dataset):,} ảnh) | "
        f"SSC={LAMBDA_SSC} | CPM={LAMBDA_CPM} ({CPM_CAM_METHOD}, start ep {CPM_START_EPOCH})"
    )
    print("=" * 88)

    for epoch in range(1, NUM_EPOCHS + 1):
        # Cập nhật CPM pseudo-masks bằng LayerCAM + SAM tại các mốc epoch chỉ định
        if epoch in CPM_REFRESH_EPOCHS and LAMBDA_CPM > 0.0:
            cpm_stats = generate_cpm_pseudo_masks(
                model=s2c_model,
                predictor=predictor,
                voc_root=voc_root,
                labels_df=df_train,
                output_dir=cpm_mask_dir,
                classes=CLASSES,
                image_size=IMAGE_SIZE,
                cam_method=CPM_CAM_METHOD,
                layercam_layers=LAYERCAM_LAYERS,
                shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
                cam_scales=CPM_CAM_SCALES,
                cam_hflip=CPM_CAM_HFLIP,
                sam_params=CPM_SAM_PARAMS,
                ambiguous_cam_thr=CPM_AMBIGUOUS_CAM_THR,
                overwrite=True,
                desc=f"[Epoch {epoch:02d}] Refresh CPM ({CPM_CAM_METHOD} -> SAM)",
            )
            print(f"[Epoch {epoch:02d}] CPM pseudo-masks updated: {cpm_stats}")

        cur_lambda_cpm = LAMBDA_CPM if epoch >= CPM_START_EPOCH else 0.0
        current_lr = optimizer.param_groups[0]["lr"]

        train_stats = train_s2c_one_epoch(
            model=s2c_model,
            proj_head=proj_head,
            loader=s2c_loader,
            optimizer=optimizer,
            cls_criterion=criterion,
            device=device,
            scheduler=scheduler,
            lambda_ssc=LAMBDA_SSC,
            lambda_cpm=cur_lambda_cpm,
            ssc_temperature=SSC_TEMPERATURE,
            use_amp=USE_AMP,
            desc=f"[S2C] Ep {epoch:02d}/{NUM_EPOCHS:02d} (cpm_w={cur_lambda_cpm:.2f})",
        )

        val_probs, val_targets, val_loss = predict_probs(
            s2c_model, val_loader, device, criterion=criterion,
            desc=f"[S2C] Ep {epoch:02d} [Val]", leave=False,
        )
        val_cls = compute_classification_metrics(val_targets, val_probs, threshold=CLS_THRESHOLD)
        val_map = val_cls["mAP"]
        val_macro_f1 = val_cls["macro_f1"]

        row = {
            "backbone": "resnet101_s2c",
            "epoch": epoch,
            "backbone_lr": current_lr,
            "train_loss": round(train_stats["loss"], 6),
            "cls_loss": round(train_stats["cls_loss"], 6),
            "ssc_loss": round(train_stats["ssc_loss"], 6),
            "cpm_loss": round(train_stats["cpm_loss"], 6),
            "val_loss": round(val_loss, 6),
            "val_mAP": round(val_map, 6),
            "val_macro_f1": round(val_macro_f1, 6),
        }
        history.append(row)

        if epoch >= CPM_START_EPOCH:
            improved = val_map > best_s2c_map
            if improved:
                best_s2c_map = val_map
                best_epoch = epoch
        else:
            improved = (best_s2c_map < 0) and (val_map > best_any_map)
            if val_map > best_any_map:
                best_any_map = val_map
                if best_s2c_map < 0:
                    best_epoch = epoch

        ckpt_payload = {
            "backbone": MODEL_NAME,
            "variant": "resnet101_s2c",
            "epoch_in_run": epoch,
            "best_epoch": best_epoch,
            "classes": list(CLASSES),
            "model_state_dict": s2c_model.state_dict(),
            "s2c_proj_state_dict": proj_head.state_dict(),
            "val_loss": val_loss,
            "val_mAP": val_map,
            "history": list(history),
            "config": config_record,
        }
        torch.save(ckpt_payload, s2c_dir / "last.pt")
        if improved:
            torch.save(ckpt_payload, s2c_best_path)
            torch.save(s2c_model.state_dict(), s2c_weights_path)

        (s2c_dir / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
        pd.DataFrame(history).to_csv(s2c_dir / "training_history.csv", index=False)

        star = " [BEST]" if improved else ""
        print(
            f"[S2C] Ep {epoch:02d}/{NUM_EPOCHS:02d} | LR: {current_lr:.2e} | "
            f"Loss: {train_stats['loss']:.4f} (cls={train_stats['cls_loss']:.4f}, "
            f"ssc={train_stats['ssc_loss']:.4f}, cpm={train_stats['cpm_loss']:.4f}) | "
            f"Val mAP: {val_map:.4f} | F1: {val_macro_f1:.4f}{star}"
        )

    trained_models["resnet101_s2c"] = load_classifier(
        MODEL_NAME, s2c_best_path, device=device, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    )
    all_histories["resnet101_s2c"] = history
    del s2c_model, proj_head, optimizer, scheduler, s2c_loader
    gc.collect()
    torch.cuda.empty_cache()

# ---------------------------------------------------------------------
# 6B. Baseline ResNet-101 (đối chứng)
# ---------------------------------------------------------------------
base_ckpt = _find_baseline_ckpt()
if base_ckpt is not None:
    trained_models["resnet101_baseline"] = load_classifier(
        MODEL_NAME, base_ckpt, device=device, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    )
    all_histories["resnet101_baseline"] = _load_history(base_ckpt)
    print(f"[Baseline] Nạp ResNet-101 baseline từ {base_ckpt}")
elif TRAIN_BASELINE_IF_MISSING:
    base_dir = ckpt_dir / "resnet101_baseline"
    base_best = base_dir / "best.pt"
    base_loader = make_loader(
        base_train_dataset, BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, device=device, seed=SEED,
    )
    base_model = build_classifier(
        model_name=MODEL_NAME, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    ).to(device)
    optimizer = build_optimizer(base_model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    scheduler = poly_scheduler(optimizer, total_steps=NUM_EPOCHS * len(base_loader), power=POLY_POWER)
    best_map = -1.0
    history_base = []

    print("\n" + "=" * 80)
    print(f"TRAIN BASELINE RESNET-101 (BCE thuần) | {NUM_EPOCHS} epochs")
    print("=" * 80)
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(
            base_model, base_loader, optimizer, criterion, device, scheduler=scheduler,
            desc=f"[Baseline] Ep {epoch:02d}/{NUM_EPOCHS:02d}",
        )
        val_probs, val_targets, val_loss = predict_probs(base_model, val_loader, device, criterion=criterion, leave=False)
        val_cls = compute_classification_metrics(val_targets, val_probs, threshold=CLS_THRESHOLD)
        val_map = val_cls["mAP"]
        history_base.append({
            "backbone": "resnet101_baseline",
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_loss": round(val_loss, 6),
            "val_mAP": round(val_map, 6),
            "val_macro_f1": round(val_cls["macro_f1"], 6),
        })
        if val_map > best_map:
            best_map = val_map
            torch.save({
                "backbone": MODEL_NAME,
                "model_state_dict": base_model.state_dict(),
                "history": list(history_base),
                "config": config_record,
            }, base_best)
        print(f"[Baseline] Ep {epoch:02d}/{NUM_EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val mAP: {val_map:.4f}")

    trained_models["resnet101_baseline"] = load_classifier(
        MODEL_NAME, base_best, device=device, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    )
    all_histories["resnet101_baseline"] = history_base
    del base_model, optimizer, scheduler, base_loader
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("[Baseline] Bỏ qua đối chứng baseline (không tìm thấy checkpoint).")

# Vẽ training curves cho S2C
s2c_hist = all_histories.get("resnet101_s2c", [])
if s2c_hist:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    ep = [r["epoch"] for r in s2c_hist]
    axes[0].plot(ep, [r.get("cls_loss", r["train_loss"]) for r in s2c_hist], label="L_cls")
    if "ssc_loss" in s2c_hist[0]:
        axes[0].plot(ep, [r["ssc_loss"] for r in s2c_hist], label="L_ssc")
        axes[0].plot(ep, [r["cpm_loss"] for r in s2c_hist], label="L_cpm")
    axes[0].set_title("S2C Loss Components"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

    for variant_name, h in all_histories.items():
        if h:
            axes[1].plot([r["epoch"] for r in h], [r["val_mAP"] * 100 for r in h], marker="o", label=variant_name)
            axes[2].plot([r["epoch"] for r in h], [r["val_macro_f1"] * 100 for r in h], marker="s", label=variant_name)
    axes[1].set_title("Val Classification mAP (%)"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].set_title("Val Macro F1 (%)"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(plots_dir / "s2c_training_curves.png", dpi=160, bbox_inches="tight")
    plt.show()


## 7. Cache SAM Embedding trên tập Tune (`TUNE_SUBSET_SIZE` ảnh cân bằng từ `Seg/train`)


In [ ]:
shared_tune_images = load_eval_samples(
    tune_balanced_dataset,
    image_size=None,
    desc=f"Loading Tune Images & Native GT Masks ({len(tune_balanced_dataset)} ảnh)",
)

tune_step = max(1, len(shared_tune_images) // TUNE_SUBSET_SIZE)
tune_indices = list(range(0, len(shared_tune_images), tune_step))[:TUNE_SUBSET_SIZE]
tune_samples = [shared_tune_images[i] for i in tune_indices]

shared_sam_embeds = {}
for local_i, sample in enumerate(tqdm(
    tune_samples, desc=f"Caching Shared SAM ({SAM_MODEL_TYPE}) Embeddings ({len(tune_samples)} ảnh)"
)):
    predictor.set_image(sample["img_u8"])
    shared_sam_embeds[local_i] = get_sam_embedding(predictor)

print(f"Tune samples: {len(tune_samples)} | SAM embeddings: {len(shared_sam_embeds)}")


## 8. Optuna Tune 4 phương pháp cho `resnet101_s2c` & `resnet101_baseline`


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_best_params = {}
id_to_seg_idx = {img_id: i for i, img_id in enumerate(seg_train_dataset.image_ids)}
required_optuna_keys = (
    "tune_protocol_version",
    "best_cls_threshold",
    "best_cam_params",
    "best_sam_params",
    "best_layercam_params",
    "best_layercam_sam_params",
)


def _normalize_sam_score_weights(w_cam_iou, w_core_recall, w_sam_iou, w_bg_spill):
    total = max(float(w_cam_iou + w_core_recall + w_sam_iou + w_bg_spill), 1e-8)
    return {
        "w_cam_iou": round(float(w_cam_iou) / total, 5),
        "w_core_recall": round(float(w_core_recall) / total, 5),
        "w_sam_iou": round(float(w_sam_iou) / total, 5),
        "w_bg_spill": round(float(w_bg_spill) / total, 5),
    }


def _sam_params_for_eval(params):
    cam_threshold = float(params["cam_threshold"])
    low_threshold = round(float(params["low_ratio"]) * cam_threshold, 4)
    bg_threshold = (
        round(float(params.get("bg_ratio", 0.0)) * low_threshold, 4) if USE_NEG_POINTS else 0.0
    )
    return {
        "cam_threshold": cam_threshold,
        "low_threshold": low_threshold,
        "bg_threshold": bg_threshold,
        "box_pad": int(params["box_pad"]),
        "cam_pixel_weight": float(params["cam_pixel_weight"]),
        "use_neg_points": USE_NEG_POINTS,
        **_normalize_sam_score_weights(
            w_cam_iou=params["w_cam_iou"],
            w_core_recall=params["w_core_recall"],
            w_sam_iou=params["w_sam_iou"],
            w_bg_spill=params["w_bg_spill"],
        ),
    }


for variant_name, model in trained_models.items():
    v_param_file = optuna_dir / f"{variant_name}_best_params.json"
    if RESUME_IF_EXISTS and v_param_file.is_file():
        loaded_params = json.loads(v_param_file.read_text(encoding="utf-8"))
        if (
            all(k in loaded_params for k in required_optuna_keys)
            and loaded_params.get("sam_model") == SAM_MODEL_TYPE
            and loaded_params.get("tune_protocol_version") == TUNE_PROTOCOL_VERSION
            and loaded_params.get("layer4_stride1") == LAYER4_STRIDE_1
            and list(loaded_params.get("layercam_layers", [])) == list(LAYERCAM_LAYERS)
        ):
            print(f"[Resume] Nạp Optuna bundle '{variant_name}' từ {v_param_file}")
            all_best_params[variant_name] = loaded_params
            continue

    print("\n" + "=" * 80)
    print(f"OPTUNA 4 PHƯƠNG PHÁP CHO: {variant_name.upper()}")
    print("=" * 80)

    seg_probs_all, seg_targets_all, _ = predict_probs(
        model, seg_train_loader, device, desc=f"[{variant_name}] Infer Seg/Train Probs"
    )

    def obj_cls(trial):
        thr = trial.suggest_float("cls_threshold", 0.25, 0.75, step=0.01)
        return compute_classification_metrics(seg_targets_all, seg_probs_all, threshold=thr)["macro_f1"]

    study_cls = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study_cls.optimize(obj_cls, n_trials=OPTUNA_CLS_TRIALS)
    best_cls_thr = study_cls.best_params["cls_threshold"]
    print(f"[{variant_name}] Best cls_threshold = {best_cls_thr:.2f} | Macro-F1 = {study_cls.best_value:.4f}")

    tune_probs = np.stack([seg_probs_all[id_to_seg_idx[item["img_id"]]] for item in tune_samples])
    cam_caches = cache_cams(
        tune_samples,
        tune_probs,
        cls_threshold=best_cls_thr,
        cam_fns={
            "cam": make_cam_fn(model, method="cam", scales=CAM_SCALES, hflip=CAM_HFLIP),
            "layercam": make_cam_fn(
                model, method="layercam", scales=CAM_SCALES, hflip=CAM_HFLIP,
                layers=LAYERCAM_LAYERS, shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
            ),
        },
        desc=f"[{variant_name}] Caching Multi-Scale CAM & LayerCAM",
    )

    def _run_cam_study(cache_list, n_trials, csv_name, label):
        def _obj(trial):
            cam_thr = trial.suggest_float("cam_threshold", 0.06, 0.45, step=0.01)
            evaluator = SegmentationEvaluator(num_classes=21, ignore_index=255)
            for base_item, cam_item in zip(tune_samples, cache_list):
                pseudo_mask = gen_cam_pseudo_mask(
                    cam_item["cams"], cam_item["active_classes"], cam_threshold=cam_thr
                )
                evaluator.update(pseudo_mask, base_item["gt_mask"])
            return evaluator.compute()["mIoU"]

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_obj, n_trials=n_trials, show_progress_bar=True)
        study.trials_dataframe().to_csv(optuna_dir / csv_name, index=False)
        print(f"[{variant_name} | {label}] Best: {study.best_params} | Tune mIoU = {study.best_value:.4f}")
        return study

    def _run_sam_study(cache_list, n_trials, csv_name, label):
        def _obj(trial):
            cam_threshold = trial.suggest_float("cam_threshold", 0.08, 0.38, step=0.02)
            low_ratio = trial.suggest_float("low_ratio", 0.25, 0.90, step=0.05)
            low_threshold = round(low_ratio * cam_threshold, 4)
            bg_ratio = trial.suggest_float("bg_ratio", 0.05, 0.60, step=0.05) if USE_NEG_POINTS else 0.0
            bg_threshold = round(bg_ratio * low_threshold, 4)
            box_pad = trial.suggest_int("box_pad", 4, 64, step=4)
            w_cam_iou = trial.suggest_float("w_cam_iou", 0.20, 0.70, step=0.05)
            w_core_recall = trial.suggest_float("w_core_recall", 0.05, 0.55, step=0.05)
            w_sam_iou = trial.suggest_float("w_sam_iou", 0.10, 0.40, step=0.05)
            w_bg_spill = trial.suggest_float("w_bg_spill", 0.20, 1.00, step=0.05)
            cam_pixel_weight = trial.suggest_float("cam_pixel_weight", 0.35, 0.90, step=0.05)
            score_weights = _normalize_sam_score_weights(
                w_cam_iou=w_cam_iou, w_core_recall=w_core_recall,
                w_sam_iou=w_sam_iou, w_bg_spill=w_bg_spill,
            )

            evaluator = SegmentationEvaluator(num_classes=21, ignore_index=255)
            for local_i, base_item in enumerate(tune_samples):
                cam_item = cache_list[local_i]
                set_sam_embedding(predictor, shared_sam_embeds[local_i], device=device)
                pseudo_mask = gen_sam_pseudo_mask(
                    predictor=predictor,
                    image_rgb=base_item["img_rgb"],
                    cams=cam_item["cams"],
                    class_indices=cam_item["active_classes"],
                    cam_threshold=cam_threshold,
                    low_threshold=low_threshold,
                    bg_threshold=bg_threshold,
                    use_neg_points=USE_NEG_POINTS,
                    box_pad=box_pad,
                    encode_image=False,
                    w_cam_iou=score_weights["w_cam_iou"],
                    w_core_recall=score_weights["w_core_recall"],
                    w_sam_iou=score_weights["w_sam_iou"],
                    w_bg_spill=score_weights["w_bg_spill"],
                    cam_pixel_weight=cam_pixel_weight,
                )
                evaluator.update(pseudo_mask, base_item["gt_mask"])
            return evaluator.compute()["mIoU"]

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_obj, n_trials=n_trials, show_progress_bar=True)
        study.trials_dataframe().to_csv(optuna_dir / csv_name, index=False)
        print(f"[{variant_name} | {label}] Best: {study.best_params} | Tune mIoU = {study.best_value:.4f}")
        return study

    study_cam = _run_cam_study(
        cam_caches["cam"], OPTUNA_CAM_TRIALS, f"{variant_name}_trials_cam.csv", "Vanilla CAM"
    )
    study_sam = _run_sam_study(
        cam_caches["cam"], OPTUNA_SAM_TRIALS, f"{variant_name}_trials_sam.csv", f"CAM + SAM ({SAM_MODEL_TYPE})"
    )
    study_layercam = _run_cam_study(
        cam_caches["layercam"], OPTUNA_LAYERCAM_TRIALS, f"{variant_name}_trials_layercam.csv", "LayerCAM"
    )
    study_layercam_sam = _run_sam_study(
        cam_caches["layercam"], OPTUNA_LAYERCAM_SAM_TRIALS,
        f"{variant_name}_trials_layercam_sam.csv", f"LayerCAM + SAM ({SAM_MODEL_TYPE})",
    )

    bundle = {
        "variant": variant_name,
        "backbone": MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "tune_protocol_version": TUNE_PROTOCOL_VERSION,
        "tune_subset_size": len(tune_samples),
        "eval_protocol": "original_resolution",
        "use_neg_points": USE_NEG_POINTS,
        "layer4_stride1": LAYER4_STRIDE_1,
        "sam_model": SAM_MODEL_TYPE,
        "cam_scales": list(CAM_SCALES),
        "cam_hflip": CAM_HFLIP,
        "layercam_layers": list(LAYERCAM_LAYERS),
        "layercam_shallow_gamma": LAYERCAM_SHALLOW_GAMMA,
        "best_cls_threshold": best_cls_thr,
        "best_cam_params": study_cam.best_params,
        "best_sam_params": _sam_params_for_eval(study_sam.best_params),
        "best_layercam_params": study_layercam.best_params,
        "best_layercam_sam_params": _sam_params_for_eval(study_layercam_sam.best_params),
        "tune_best_miou": {
            "vanilla_cam": round(float(study_cam.best_value), 4),
            "cam_sam": round(float(study_sam.best_value), 4),
            "layercam": round(float(study_layercam.best_value), 4),
            "layercam_sam": round(float(study_layercam_sam.best_value), 4),
        },
    }
    v_param_file.write_text(json.dumps(bundle, indent=2, ensure_ascii=False), encoding="utf-8")
    all_best_params[variant_name] = bundle

    del cam_caches
    gc.collect()
    torch.cuda.empty_cache()

shared_sam_embeds.clear()
gc.collect()
torch.cuda.empty_cache()


## 9. Đánh giá cuối trên `Segmentation/val.txt` (1,449 ảnh, độ phân giải gốc)

So sánh trực tiếp **`resnet101_baseline`** vs **`resnet101_s2c`** trên 4 phương pháp (`Vanilla CAM`, `CAM + SAM`, `LayerCAM`, `LayerCAM + SAM`).


In [ ]:
val_probs_by_variant = {}
cls_summary_rows = []
val_targets_all = None

for variant_name, model in trained_models.items():
    cls_thr = all_best_params[variant_name]["best_cls_threshold"]
    probs_arr, targets_arr, _ = predict_probs(
        model, val_loader, device, desc=f"Val Classification (1,449 ảnh) [{variant_name}]"
    )
    if val_targets_all is None:
        val_targets_all = targets_arr
    val_probs_by_variant[variant_name] = probs_arr

    cls_metrics = compute_classification_metrics(val_targets_all, probs_arr, threshold=cls_thr)
    log_eval_results(
        experiment_name=f"{variant_name}_Classification",
        metrics=cls_metrics,
        config={**config_record, "variant": variant_name, "cls_threshold": cls_thr},
        log_dir=run_dir,
    )
    cls_summary_rows.append(
        cls_summary_row(cls_metrics, percent=True, Variant=variant_name, **{"Best Cls Thr": cls_thr})
    )

cls_summary_df = pd.DataFrame(cls_summary_rows)
cls_summary_df.to_csv(run_dir / "val_classification_summary.csv", index=False)
display(cls_summary_df)

pipelines = {
    variant_name: build_optuna_pipeline(
        model=trained_models[variant_name],
        probs=val_probs_by_variant[variant_name],
        best_params=all_best_params[variant_name],
        predictor=predictor,
        encode_image=False,
        scales=CAM_SCALES,
        hflip=CAM_HFLIP,
        layercam_layers=LAYERCAM_LAYERS,
        shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
    )
    for variant_name in trained_models
}

computed_seg_metrics = evaluate_segmentation(
    val_dataset,
    pipelines,
    image_size=None,
    prepare_fn=predictor.set_image,
    desc="Val Segmentation (1,449 ảnh — Baseline vs S2C × 4 methods)",
)

seg_summary_rows = []
per_class_records = {"Class": ["background", *CLASSES]}
for variant_name in trained_models:
    for method_label in METHODS:
        metrics = computed_seg_metrics[variant_name][method_label]
        safe_method = method_label.replace(" ", "").replace("+", "_")
        log_eval_results(
            experiment_name=f"{variant_name}_{safe_method}",
            metrics=metrics,
            config={**all_best_params[variant_name], "method": method_label},
            log_dir=run_dir,
        )
        seg_summary_rows.append(
            seg_summary_row(metrics, percent=True, Variant=variant_name, Method=method_label)
        )
        per_class_records[f"{variant_name} | {method_label}"] = [
            round(v * 100, 2) for v in metrics["iou_per_class"]
        ]

seg_df = pd.DataFrame(seg_summary_rows)
seg_df.to_csv(run_dir / "val_segmentation_summary.csv", index=False)
pd.DataFrame(per_class_records).to_csv(run_dir / "val_per_class_iou.csv", index=False)

print("\n=== SO SÁNH PHÂN ĐOẠN TRÊN VAL (1,449 ẢNH): BASELINE vs S2C ===")
display(seg_df)

variants_list = list(trained_models.keys())
fig, ax = plt.subplots(figsize=(11.5, 5.4))
x = np.arange(len(variants_list))
width = 0.2
method_colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]
for idx, method_label in enumerate(METHODS):
    values = [computed_seg_metrics[v][method_label]["mIoU"] * 100 for v in variants_list]
    bars = ax.bar(
        x + (idx - (len(METHODS) - 1) / 2) * width,
        values, width=width, label=method_label, color=method_colors[idx],
    )
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([v.upper() for v in variants_list], fontsize=11, fontweight="bold")
ax.set_ylabel("Val mIoU (%)")
ax.set_title(f"ResNet-101 Baseline vs S2C ({CPM_CAM_METHOD} CPM) — SAM {SAM_MODEL_TYPE}")
ax.set_ylim(0, seg_df["mIoU (%)"].max() + 8)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.10))
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(plots_dir / "val_miou_baseline_vs_s2c.png", dpi=160, bbox_inches="tight")
plt.show()


## 10. Trực quan hoá định tính (Baseline vs S2C CAM / LayerCAM / SAM)


In [ ]:
sample_indices = [0, 12, 25, 42]
rows = []
for idx in sample_indices:
    sample = load_eval_sample(val_dataset, idx, image_size=None)
    predictor.set_image(sample["img_u8"])

    s2c_pipe = pipelines["resnet101_s2c"]
    s2c_out = predict_sample_masks(sample, s2c_pipe, s2c_pipe.probs[idx])
    active_names = [CLASSES[c] for c in s2c_out["active_classes"]]

    row = [
        image_cell(sample["img_rgb"], f"{sample['img_id']}\n{active_names}"),
        mask_cell(sample["gt_mask"], "Ground Truth"),
    ]
    if "resnet101_baseline" in pipelines:
        base_pipe = pipelines["resnet101_baseline"]
        base_out = predict_sample_masks(sample, base_pipe, base_pipe.probs[idx])
        row.extend([
            mask_cell(base_out["masks"]["Vanilla CAM"], "Base CAM"),
            mask_cell(base_out["masks"]["LayerCAM"], "Base LayerCAM"),
        ])
    row.extend([
        mask_cell(s2c_out["masks"]["Vanilla CAM"], "S2C CAM"),
        mask_cell(s2c_out["masks"]["LayerCAM"], "S2C LayerCAM"),
        mask_cell(s2c_out["masks"]["LayerCAM + SAM"], "S2C LCAM+SAM"),
    ])
    rows.append(row)

plot_grid(rows, cell_size=3.0, save_path=plots_dir / "val_qualitative_s2c.png")

print("=" * 80)
print(f"HOÀN TẤT — kết quả lưu tại: {run_dir}")
print("=" * 80)
if "seg_df" in globals() and len(seg_df) > 0:
    best_row = seg_df.sort_values("mIoU (%)", ascending=False).iloc[0]
    print(f"Best mIoU: {best_row['mIoU (%)']:.2f}% | {best_row['Variant']} | {best_row['Method']}")
